# Heart Disease Across Hospitals — Generated Evidence

This notebook is a thin, executable report. It reads outputs produced by the installable `heart_disease` package and intentionally contains no second training implementation.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'pyproject.toml').is_file()
)
REPORTS = PROJECT_ROOT / 'reports'
metrics = json.loads((REPORTS / 'metrics.json').read_text(encoding='utf-8'))
external = pd.read_csv(REPORTS / 'external-validation.csv')
manifest = json.loads((REPORTS / 'experiment-manifest.json').read_text(encoding='utf-8'))

## Reproduction identity

In [ ]:
display(Markdown(
    f"**Profile:** `{metrics['profile']}`  \\n"
    f"**Selected model:** `{metrics['selection']['model_name']}`  \\n"
    f"**Source commit:** `{manifest['git_commit']}`"
))

## Nested development evidence

In [ ]:
candidate_table = pd.DataFrame(metrics['candidates']).T[[
    'mean_roc_auc', 'std_roc_auc', 'mean_balanced_accuracy', 'mean_brier_score'
]]
candidate_table.round(3)

## Frozen external validation

In [ ]:
external.loc[external['threshold_name'].eq('default'), [
    'cohort', 'n', 'prevalence', 'roc_auc', 'roc_auc_ci_low',
    'roc_auc_ci_high', 'balanced_accuracy', 'brier_score'
]]

## Diagnostic figures

In [ ]:
for filename in [
    'roc-curve.png', 'precision-recall-curve.png', 'calibration-curve.png',
    'confusion-matrices.png', 'missingness.png', 'cohort-shift.png',
    'feature-stability.png',
]:
    display(Markdown(f'### {filename.removesuffix(".png").replace("-", " " ).title()}'))
    display(Image(filename=str(REPORTS / 'figures' / filename), width=760))

## Unsupervised patient-profile discovery

This section consumes the generated label-isolated PCA and clustering evidence; it does not fit or select clusters in the notebook.

In [ ]:
UNSUPERVISED_SUMMARY = PROJECT_ROOT / 'reports/unsupervised/summary.json'
UNSUPERVISED_SELECTION = PROJECT_ROOT / 'reports/unsupervised/cluster-selection.csv'
UNSUPERVISED_PROFILES = PROJECT_ROOT / 'reports/unsupervised/cluster-profiles.csv'
UNSUPERVISED_TRANSFER = PROJECT_ROOT / 'reports/unsupervised/external-transfer.csv'

if UNSUPERVISED_SUMMARY.is_file():
    unsupervised_summary = json.loads(UNSUPERVISED_SUMMARY.read_text(encoding='utf-8'))
    cluster_selection = pd.read_csv(UNSUPERVISED_SELECTION)
    cluster_profiles = pd.read_csv(UNSUPERVISED_PROFILES)
    external_transfer = pd.read_csv(UNSUPERVISED_TRANSFER)
    display(Markdown(
        f"**Selected k:** `{unsupervised_summary['selected_k']}`  \\n"
        f"**Retained PCA components:** `{unsupervised_summary['retained_components']}`"
    ))
    display(cluster_selection.round(3))
    display(cluster_profiles)
    display(external_transfer.round(3))
else:
    display(Markdown(
        'Generate this evidence with `heart-disease analyze-unsupervised --profile full`.'
    ))

In [ ]:
for filename in [
    'pca-explained-variance.png', 'pca-clusters.png',
    'cluster-selection.png', 'hierarchical-dendrogram.png',
    'cluster-profiles.png',
]:
    figure_path = REPORTS / 'unsupervised' / 'figures' / filename
    if figure_path.is_file():
        title = filename.removesuffix('.png').replace('-', ' ').title()
        display(Markdown(f'### Exploratory {title}'))
        display(Image(filename=str(figure_path), width=760))

### Unsupervised interpretation limits

Disease labels are joined only after PCA and clustering choices are frozen. Cluster IDs are arbitrary, mixed-data Euclidean geometry is a pragmatic limitation, and the groups are exploratory—not diagnoses or clinical subtypes.

## Interpretation limit

The decline across external hospitals is part of the result. These historical cohorts do not establish present-day clinical utility. This project is not medical advice and is not clinically validated.